In [1]:
import pathlib

from examples.get_accounts import client_secret

from mecon.etl.dataset import DatasetDir
from mecon.app.current_data import WorkingDatasetDir
from mecon.tags.process import RuleExecutionPlanMonitor
from mecon.data.data_management import CachedFileDataManager
from mecon.tags import process
import pandas as pd
from mecon.tags import tagging
from pprint import pprint
from datetime import datetime
import requests

# datasets = WorkingDatasetDir()
datasets = DatasetDir(path=r"/Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets")
datasets.dataset_names()

Failed to retrieve accounts


INFO:root:MECON_ROOT_DIRPATH not found in environment variables. Using default value relative to __file__='/Users/wimpole/PycharmProjects/mecon/mecon/config.py'.
INFO:root:MECON_ROOT_DIRPATH set to: MECON_ROOT_DIRPATH=PosixPath('/Users/wimpole/PycharmProjects/mecon')
INFO:root:Adding 14 datasets from /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets. #info#filesystem
INFO:root:New dataset in path '/Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/shared_backup'. #info#filesystem
INFO:root:New dataset in path '/Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/test'. #info#filesystem
INFO:root:New dataset in path '/Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/v2_dataset_old_ids'. #info#filesystem
INFO:root:New dataset in path '/Users/wimpole/Library/CloudStorage/GoogleDri

['shared_backup',
 'test',
 'v2_dataset_old_ids',
 'mydata',
 'shared_monzoapi',
 'shared',
 'mydata_backup',
 'v2',
 'v2_dataset_test',
 'v3',
 'v2_dataset_true_layer',
 'v2_backup',
 'v2_dataset_test_monzo_api',
 '.idea']

In [2]:
dataset = datasets.get_dataset('v2_dataset_true_layer')
dm = CachedFileDataManager(dataset)


In [3]:
import json
from mecon.settings import DictFile

creds_file = datasets.path / 'credentials.json'
# creds = json.loads(creds_file.read_text())
creds = DictFile(creds_file)

In [17]:

import webbrowser

client_id = creds['client_id']
client_secret = creds['client_secret']
redirect_uri = creds['redirect_uri']

auth_url = f"""
https://auth.truelayer.com/?response_type=code
&client_id={client_id}
&scope=info%20accounts%20transactions%20offline_access
&redirect_uri={redirect_uri}
&providers=revolut
&state=abc123
&nonce=xyz456
""".replace('\n', '')

webbrowser.open(auth_url)


True

In [13]:
bank = 'hsbc'
tl_creds = creds['truelayer']
data = {
        'grant_type': 'authorization_code',
        'client_id': tl_creds['client_id'],
        'client_secret': tl_creds['client_secret'],
        'redirect_uri': tl_creds['redirect_uri'],
        'code': tl_creds['sources'][bank]['authentication_code']['code'],
    }

print(data)

token_res = requests.post('https://auth.truelayer.com/connect/token', data=data)

print(f"{token_res.status_code=} {token_res.text=}")



{'grant_type': 'authorization_code', 'client_id': 'mecon-e3786b', 'client_secret': 'd729aef4-eea1-49c7-af46-4468ac0a2005', 'redirect_uri': 'https://console.truelayer.com/redirect-page', 'code': '648FE9E5D006406EFAB64406BF57B3DCBA96CC28FC1489A5D3462092650571DA'}
token_res.status_code=400 token_res.text='{"error":"invalid_client"}'


In [45]:
import requests

def fetch_access_token(bank):
    data = {
        'grant_type': 'authorization_code',
        'client_id': creds['client_id'],
        'client_secret': creds['client_secret'],
        'redirect_uri': redirect_uri,
        'code': creds[bank]['authentication_code'],
    }

    token_res = requests.post('https://auth.truelayer.com/connect/token', data=data)

    if token_res.status_code != 200:
        print(f"Error fetching access token: {token_res.status_code}")
    else:
        creds[bank]['token'] = token_res.json()
        print(f"Error fetching access token: {token_res.status_code}")

        # pprint(creds['hsbc']['token'])

def refresh_token(bank):
    url = "https://auth.truelayer.com/connect/token"

    data = {
        "grant_type": "refresh_token",
        "client_id": creds['client_id'],
        "client_secret": creds['client_secret'],
        "refresh_token": creds[bank]['token']['refresh_token'],
    }

    headers = {
        "accept": "application/json",
        "content-type": "application/x-www-form-urlencoded"
    }

    response = requests.post(url, headers=headers, data=data)

    if response.status_code != 200:
        print("❌", response.status_code, response.text)
    else:
        new_token = response.json()
        # Add timestamp
        new_token['refreshed_at'] = datetime.utcnow().isoformat() + "Z"

        # Save it back into your creds structure
        creds[bank]['token'] = new_token
        creds.save()
        print(f"✅Token for '{bank.upper()}' successfully refreshed")

def reauthenticate(bank):
    url = "https://auth.truelayer.com/v1/reauthuri"

    payload = {
        "response_type": "code",
        "refresh_token": creds[bank]['token']['refresh_token'],
        'redirect_uri': creds['redirect_uri'],

    }
    headers = {
        "accept": "application/json",
        "content-type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)

    if response.status_code == 200:
        print(f"✅ Successfully refreshed!")
    else:
        print(f"❌ Error {response.status_code}: {response.text}")




def fetch_accounts(bank):
    headers = {
        'Authorization': f"Bearer {creds[bank]['token']['access_token']}"
    }
    acc_res = requests.get('https://api.truelayer.com/data/v1/accounts', headers=headers)
    if acc_res.status_code != 200:
        print(acc_res.text)
        return []
    else:
        creds[bank]['accounts'] = acc_res.json()['results']
        creds.save()
        print(f"Accounts for '{bank.upper()}' successfully refreshed")
        for acc in creds[bank]['accounts']:
            print(f"{acc['account_id']} | {acc['account_type']} | {acc['display_name']}")
        return [acc['account_id'] for acc in creds[bank]['accounts']]

def fetch_transactions(bank, account_id, file_pointer: pathlib.Path=None):
    headers = {
        'Authorization': f"Bearer {creds[bank]['token']['access_token']}",
        "accept": "application/json; charset=UTF-8"
    }

    # from_date = (datetime.date.today() - datetime.timedelta(days=180)).isoformat()
    # to_date = datetime.date.today().isoformat()
    url = f"https://api.truelayer.com/data/v1/accounts/{account_id}/transactions?from=2023-06-01&to=2025-06-2"
    # url = f"https://api.truelayer.com/data/v1/accounts/{account_id}/transactions"

    txn_res = requests.get(url, headers=headers)

    if txn_res.status_code == 200:
        txns = txn_res.json()['results']
        if file_pointer is not None:
            file_pointer.parent.mkdir(parents=True, exist_ok=True)
            file_pointer.write_text(json.dumps(txns, indent=4))
            print(f"✅ Got {len(txns)} transactions and saved to {file_pointer}")
        else:
            print(f"✅ Got {len(txns)} transactions")
        return txns
    else:
        print(f"❌ Error {txn_res.status_code}: {txn_res.text}")
        return None




# HSBC

In [29]:
# fetch_access_token('hsbc')
refresh_token('hsbc')

INFO:root:Saving settings to /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/tl_credentials.json


✅Token for 'HSBC' successfully refreshed


In [9]:
fetch_accounts('hsbc')

INFO:root:Saving settings to /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/tl_credentials.json


Accounts for 'HSBC' successfully refreshed
d4aa58643585c1e3a5f7d3e24cf5e829 | TRANSACTION | HSBC ADVANCE
875dba485407b435dfddccc5a91e772b | SAVINGS | ON BNS SAVER


['d4aa58643585c1e3a5f7d3e24cf5e829', '875dba485407b435dfddccc5a91e772b']

In [30]:
for acc_id in fetch_accounts('hsbc'):
    fetch_transactions('hsbc', acc_id, file_pointer=pathlib.Path(f"hsbc/{acc_id}/transactions.json"))

INFO:root:Saving settings to /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/tl_credentials.json


Accounts for 'HSBC' successfully refreshed
d4aa58643585c1e3a5f7d3e24cf5e829 | TRANSACTION | HSBC ADVANCE
875dba485407b435dfddccc5a91e772b | SAVINGS | ON BNS SAVER
✅ Got 128 transactions and saved to hsbc/d4aa58643585c1e3a5f7d3e24cf5e829/transactions.json
✅ Got 0 transactions and saved to hsbc/875dba485407b435dfddccc5a91e772b/transactions.json


# Revolut

In [22]:
# fetch_access_token('revolut')
refresh_token('revolut')

INFO:root:Saving settings to /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/tl_credentials.json


✅Token for 'REVOLUT' successfully refreshed


In [177]:
fetch_accounts('revolut')

INFO:root:Saving settings to /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/tl_credentials.json


Accounts for 'REVOLUT' successfully refreshed
3b2038675f58008e4e58c43a5d8d103c | TRANSACTION | DIMITRIOS KONTZEDAKIS
5f2ed9feaf603a7a7a904469f37b260a | TRANSACTION | DIMITRIOS KONTZEDAKIS
fa5ddbfc7431ffd009445263b4259094 | TRANSACTION | DIMITRIOS KONTZEDAKIS
3ea5d7076b553a642d47c90ab5efec8b | TRANSACTION | DIMITRIOS KONTZEDAKIS


['3b2038675f58008e4e58c43a5d8d103c',
 '5f2ed9feaf603a7a7a904469f37b260a',
 'fa5ddbfc7431ffd009445263b4259094',
 '3ea5d7076b553a642d47c90ab5efec8b']

In [23]:
for acc_id in fetch_accounts('revolut'):
    fetch_transactions('revolut', acc_id, file_pointer=pathlib.Path(f"revolut/{acc_id}/transactions.json"))

INFO:root:Saving settings to /Users/wimpole/Library/CloudStorage/GoogleDrive-jimitsos41@gmail.com/Other computers/My Laptop/datasets/tl_credentials.json


Accounts for 'REVOLUT' successfully refreshed
3b2038675f58008e4e58c43a5d8d103c | TRANSACTION | DIMITRIOS KONTZEDAKIS
5f2ed9feaf603a7a7a904469f37b260a | TRANSACTION | DIMITRIOS KONTZEDAKIS
fa5ddbfc7431ffd009445263b4259094 | TRANSACTION | DIMITRIOS KONTZEDAKIS
3ea5d7076b553a642d47c90ab5efec8b | TRANSACTION | DIMITRIOS KONTZEDAKIS
❌ Error 403: {"error_description":"SCA exemption has expired. This resource is protected and should be accessed shortly after PSU Authentication. In order to access this resource, please have the PSU re-authenticate.","error":"sca_exceeded","error_details":{"provider_details":"403 access_denied: SCA exemption has expired. This resource is protected and should be accessed within 5 minutes of PSU Authentication. In order to access this resource, please have the PSU re-authenticate."}}
❌ Error 403: {"error_description":"SCA exemption has expired. This resource is protected and should be accessed shortly after PSU Authentication. In order to access this resource, pl